In [18]:
# import pandas as pd
# tool_name = "vllm"
# df1 = pd.read_csv("/Users/forough/PycharmProjects/serving/dataset/repos_vllm(from_vllm).csv")
# df2 = pd.read_csv("/Users/forough/PycharmProjects/serving/dataset/repos_vllm(import_vllm).csv")
# merged_df = pd.concat([df1, df2], ignore_index=True)
# merged_df = merged_df.drop_duplicates(ignore_index=True)
# # Split into chunks of size 5000
# chunk_size = 5000
# for i in range(0, len(merged_df), chunk_size):
#     chunk = merged_df.iloc[i:i + chunk_size].copy()
#     chunk.to_csv(
#         f"/Users/forough/PycharmProjects/serving/dataset/repos_vllm_chunk_{i//chunk_size + 1}.csv",
#         index=False
#     )

In [19]:
import pandas as pd
tool_name = "flashinfer"
df1 = pd.read_csv(f"/Users/forough/VScodeProjects/VScode_serving/dataset/codes_links_and_repo_names/files_{tool_name}(from_{tool_name}).csv")
df2 = pd.read_csv(f"/Users/forough/VScodeProjects/VScode_serving/dataset/codes_links_and_repo_names/files_{tool_name}(import_{tool_name}).csv")
merged_df = pd.concat([df1, df2], ignore_index=True)
merged_df = merged_df.drop_duplicates(ignore_index=True)
print(len(merged_df))
# merged_df = merged_df.head(100).copy()
# Split into chunks of size 5000
chunk_size = 5000
for i in range(0, len(merged_df), chunk_size):
    chunk = merged_df.iloc[i:i + chunk_size].copy()
    chunk.to_csv(
                f"/Users/forough/VScodeProjects/VScode_serving/dataset/repo_info/old repo info/{tool_name}/repos_{tool_name}_chunk_{i//chunk_size + 1}.csv",
        index=False
    )

2547


In [20]:
import os
import glob
import time
import pandas as pd
import requests
from urllib.parse import urlparse


def call_github_api(query, max_retries=3, backoff_sec=2):
    """POST a GraphQL query with simple retry + token rotation."""
    for attempt in range(max_retries):
        token = get_new_token()
        header = {"Authorization": f"Bearer {token}"}

        try:
            response = requests.request(
                "POST",
                url="https://api.github.com/graphql",
                json={"query": query},
                headers=header,
                timeout=30
            )
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(backoff_sec * (attempt + 1))
                continue
            return None

        print(f"response status code is {response.status_code}")

        if response.status_code == 200:
            return response.json()

        # Handle rate limiting / transient errors with a brief backoff & retry
        if response.status_code in (429, 502, 503, 504) and attempt < max_retries - 1:
            time.sleep(backoff_sec * (attempt + 1))
            continue

        return None

    return None


token_counter = 0
def get_new_token():
    """Rotate through multiple GitHub tokens."""
    global token_counter
    tokens = [  "REDACTED_CREDENTIAL",
                "REDACTED_CREDENTIAL",
                "REDACTED_CREDENTIAL",
                "REDACTED_CREDENTIAL",
                "REDACTED_CREDENTIAL",
                "REDACTED_CREDENTIAL",
                "REDACTED_CREDENTIAL",
                "REDACTED_CREDENTIAL"]
    token = tokens[token_counter % len(tokens)]
    token_counter += 1
    return token


def extract_repo_name_owner(repo_full_name):
    repo_name_owner = repo_full_name.split('/')
    return repo_name_owner[0], repo_name_owner[1]


def make_query_issue_pr_commits(repo_full_name):
    repo_owner, repo_name = extract_repo_name_owner(repo_full_name)

    # We query multiple README name variants. We also pull open issues,
    # keywords (repositoryTopics), and last commit date.
    query = f'''
    {{
      repository(owner: "{repo_owner}", name: "{repo_name}") {{
        # About
        description

        # Counts
        issues(states: CLOSED) {{ totalCount }}
        openIssues: issues(states: OPEN) {{ totalCount }}
        pullRequests(states: MERGED) {{ totalCount }}
        stargazerCount
        forkCount

        # Default branch + commits
        defaultBranchRef {{
          name
          target {{
            ... on Commit {{
              committedDate
              history {{
                totalCount
              }}
            }}
          }}
        }}

        # Keywords (topics)
        repositoryTopics(first: 20) {{
          nodes {{ topic {{ name }} }}
        }}

        # README candidates (HEAD used; GitHub resolves it to default branch)
        readme_md: object(expression: "HEAD:README.md") {{ ...on Blob {{ text }} }}
        readme_MD: object(expression: "HEAD:README.MD") {{ ...on Blob {{ text }} }}
        readme_plain: object(expression: "HEAD:README") {{ ...on Blob {{ text }} }}
        readme_rst: object(expression: "HEAD:README.rst") {{ ...on Blob {{ text }} }}
        readme_RST: object(expression: "HEAD:README.RST") {{ ...on Blob {{ text }} }}
      }}
    }}
    '''
    return query


def _first_nonempty_readme(repo_data):
    """Pick the first available README text among common names."""
    for key in ["readme_md", "readme_MD", "readme_plain", "readme_rst", "readme_RST"]:
        val = repo_data.get(key)
        if isinstance(val, dict):
            text = val.get("text")
            if text:
                return text
    return ""


def process_chunk_csv(input_csv_path, output_csv_path):
    """Read one chunk CSV, enrich it via GitHub API, and write output CSV."""
    print(f"\nProcessing: {input_csv_path}")
    repos = pd.read_csv(input_csv_path)

    # Ensure deduplicated rows if needed (optional)
    repos = repos.drop_duplicates(ignore_index=True)

    repos_closed_issues = []
    repos_open_issues = []
    repos_merged_pr = []
    repos_commit_num = []
    repos_last_commit_date = []
    repos_stars = []
    repos_forks = []
    repos_about = []
    repos_keywords = []
    repos_readme = []

    for _, row in repos.iterrows():
        repo_full_name = row['repo_full_name']
        print(f"  -> {repo_full_name}")

        query = make_query_issue_pr_commits(repo_full_name)
        response = call_github_api(query)
        
        if response and isinstance(response, dict):
            repo_data = response.get("data", {}).get("repository", {}) or {}
        else:
            repo_data = {}


        # About
        about_text = repo_data.get("description") or ""
        repos_about.append(about_text)

        # Commits (count + last commit date)
        default_branch = repo_data.get('defaultBranchRef')
        if default_branch and isinstance(default_branch, dict):
            target = default_branch.get('target') or {}
            history = target.get('history') or {}
            commit_num = history.get('totalCount', 0)
            last_commit_date = target.get('committedDate') or ""
        else:
            commit_num = 0
            last_commit_date = ""
        repos_commit_num.append(commit_num)
        repos_last_commit_date.append(last_commit_date)

        # Issues (closed + open)
        repo_issues_closed = repo_data.get("issues", {}).get("totalCount", 0)
        repos_closed_issues.append(repo_issues_closed)

        repo_issues_open = repo_data.get("openIssues", {}).get("totalCount", 0)
        repos_open_issues.append(repo_issues_open)

        # PRs
        repo_pr = repo_data.get("pullRequests", {}).get("totalCount", 0)
        repos_merged_pr.append(repo_pr)

        # Stars / forks
        repos_stars.append(repo_data.get("stargazerCount", 0))
        repos_forks.append(repo_data.get("forkCount", 0))

        # Keywords (topics)
        topics_nodes = repo_data.get("repositoryTopics", {}).get("nodes", [])
        topics = [n.get("topic", {}).get("name") for n in topics_nodes if isinstance(n, dict)]
        repos_keywords.append(", ".join([t for t in topics if t]))

        # README (first non-empty among common names)
        repos_readme.append(_first_nonempty_readme(repo_data))

        # Optional small delay to be gentle with the API
        time.sleep(0.05)

    # Add columns and write out this chunk's enriched CSV
    repos["about"] = repos_about
    repos["keywords"] = repos_keywords
    repos["open_issues"] = repos_open_issues
    repos["closed_issues"] = repos_closed_issues
    repos["merged_pr"] = repos_merged_pr
    repos["commits"] = repos_commit_num
    repos["last_commit_date"] = repos_last_commit_date
    repos["stars"] = repos_stars
    repos["forks"] = repos_forks
    repos["readme"] = repos_readme

    repos.to_csv(output_csv_path, index=False)
    print(f"Saved: {output_csv_path}")


def run_over_all_chunks():
    # Adjust these paths/patterns if you use a different naming scheme.
    base_dir = f"/Users/forough/VScodeProjects/VScode_serving/dataset/repo_info/old repo info/{tool_name}"
    input_pattern = os.path.join(base_dir, f"repos_{tool_name}_chunk_*.csv")

    chunk_paths = sorted(glob.glob(input_pattern))

    if not chunk_paths:
        print(f"No chunk files found for pattern: {input_pattern}")
        return

    for in_path in chunk_paths:
        # Derive an output name like: info_repos_vllm_chunk_1.csv
        base = os.path.basename(in_path)  # e.g., "repos_vllm_chunk_1.csv"
        out_name = f"info_{base.replace('repos_', '')}"  # -> "info_vllm_chunk_1.csv"
        # If you prefer a fixed prefix, use:
        # out_name = base.replace("repos_", "info_repos_")
        out_path = os.path.join(base_dir, out_name)

        process_chunk_csv(in_path, out_path)


if __name__ == "__main__":
    run_over_all_chunks()


Processing: /Users/forough/VScodeProjects/VScode_serving/dataset/repo_info/old repo info/flashinfer/repos_flashinfer_chunk_1.csv
  -> aphrodite-engine/aphrodite-engine
response status code is 200
  -> AdrSuns/REAP
response status code is 200
  -> vllm-project/vllm
response status code is 200
  -> MrTang-Yuhan/Accel-Sim-Flashinfer
response status code is 200
  -> IBM/vllm
response status code is 200
  -> YelinShan/vllm-scheduling-optimized
response status code is 200
  -> Xrelifen/EdgeAI_Final
response status code is 200
  -> LINs-lab/DeFT
response status code is 200
  -> chiwanpark/tinylm
response status code is 200
  -> lambda7xx/sglang-old
response status code is 200
  -> Xrelifen/EdgeAI_Final
response status code is 200
  -> flashinfer-ai/flashinfer
response status code is 200
  -> PanZaifeng/FastTree-Artifact
response status code is 200
  -> blitz-serving/flashinfer
response status code is 200
  -> flashinfer-ai/flashinfer
response status code is 200
  -> thu-nics/MoA_Kernel
respo